# Многопроцессность
Многопроцессность — это создание нескольких независимых процессов, каждый со своим интерпретатором Python и своей памятью.

Многопроцессность — это подход, при котором программа создает несколько независимых процессов, каждый со своим интерпретатором Python, своей памятью и своим GIL. В отличие от многопоточности, где потоки разделяют память и страдают от GIL (что делает их бесполезными для CPU-задач), многопроцессность позволяет обойти GIL и реально использовать несколько ядер процессора для ускорения вычислений. Однако за это приходится платить: процессы создаются медленнее, потребляют больше памяти (каждый процесс получает копию данных), а общение между ними сложнее, чем между потоками. Поэтому многопроцессность используют для CPU-интенсивных задач (сложные расчеты, обработка данных), а для I/O-задач (сеть, диски) лучше подходят многопоточность или асинхронность. Синхронный же код — самый простой, но он выполняет задачи последовательно, без всякого ускорения.

In [13]:
from multiprocessing import Process
from multiprocessing import get_context
import multiprocessing
import os
from concurrent.futures import ProcessPoolExecutor

In [14]:
def worker(name):
    print(f"Процесс {name}: PID = {os.getpid()}")
    print(f"Процесс {name}: закончил работу")

if __name__ == "__main__":
    # Используем fork вместо spawn
    ctx = get_context('fork')
    
    p1 = ctx.Process(target=worker, args=("A",))
    p2 = ctx.Process(target=worker, args=("B",))
    
    p1.start()
    p2.start()
    
    p1.join()
    p2.join()
    
    print("Все процессы завершены")

Процесс A: PID = 79456
Процесс A: закончил работу
Процесс B: PID = 79457
Процесс B: закончил работу
Все процессы завершены


> `from multiprocessing import Process`
>
> `import os`
> 
> `def worker(name):`
>
>     print(f"Процесс {name}: PID = {os.getpid()}")
>
>     print(f"Процесс {name}: закончил работу")
> 
> `if __name__ == "__main__":`
>
>     p1 = Process(target=worker, args=("A",))
>     p2 = Process(target=worker, args=("B",))
>     
>     p1.start()
>     p2.start()
>     
>     p1.join()
>     p2.join()
>
>     print("Все процессы завершены")

Выше вариант не для Jupyter

## Упрощенный вариант

In [15]:
def worker(name):
    print(f"Процесс {name}: PID = {os.getpid()}\n", end="")
    print(f"Процесс {name}: закончил работу\n", end="")
    return name

# Принудительно устанавливаем fork
multiprocessing.set_start_method('fork', force=True)

with ProcessPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(worker, ["A", "B"]))
    print(results)

print("Все процессы завершены")

Процесс B: PID = 79459
Процесс A: PID = 79458


Процесс A: закончил работу
Процесс B: закончил работу
['A', 'B']
Все процессы завершены
